# Applications of Machine Learning to the Analysis of MD Data

MD simulations can produce large quantities of complex data, Machine Learning (ML) methods can be a great way to help make sense of it. Here we look at two of them: Principal Component Analysis (PCA) and Clustering.

We will illustrate the use of both methods to analyse and explore the same 50ns simulation of the abl+imatinib analogue complex we studied in the previous workshop, so let's begin by loading this data, plus required Python packages:

In [ ]:
import mdtraj as mdt
from matplotlib import pyplot as plt
from matplotlib.collections import LineCollection
import numpy as np
import nglview as nv

traj = mdt.load('abl_ligand_prod_dry.nc', top='abl_ligand_dry.pdb')
print(traj)

## 1. PCA for the visualization of shape space

An MD simulation is a walk through a shape space. For a system of *N* atoms, this shape space has *3N* dimensions. Each snapshot corresponds to a point in this *3N*-dimensional space. However, for visualization purposes, it would be very useful if we could somehow approximate this to a much lower dimensional space - maybe just 2 or 3 dimensions.

Principal Component Analysis (PCA) provides a method to achieve this - dimensionality reduction.

First let's create a slightly fancy graph plotting function, that will produce a nicely coloured line to track the trajectory through this space:

In [ ]:
# Taken from https://matplotlib.org/stable/gallery/lines_bars_and_markers/multicolored_line.html
def colored_line(x, y, c, ax, **lc_kwargs):
    """
    Plot a line with a color specified along the line by a third value.

    It does this by creating a collection of line segments. Each line segment is
    made up of two straight lines each connecting the current (x, y) point to the
    midpoints of the lines connecting the current point with its two neighbors.
    This creates a smooth line with no gaps between the line segments.

    Parameters
    ----------
    x, y : array-like
        The horizontal and vertical coordinates of the data points.
    c : array-like
        The color values, which should be the same size as x and y.
    ax : Axes
        Axis object on which to plot the colored line.
    **lc_kwargs
        Any additional arguments to pass to matplotlib.collections.LineCollection
        constructor. This should not include the array keyword argument because
        that is set to the color argument. If provided, it will be overridden.

    Returns
    -------
    matplotlib.collections.LineCollection
        The generated line collection representing the colored line.
    """
    if "array" in lc_kwargs:
        warnings.warn('The provided "array" keyword argument will be overridden')

    # Default the capstyle to butt so that the line segments smoothly line up
    default_kwargs = {"capstyle": "butt"}
    default_kwargs.update(lc_kwargs)

    # Compute the midpoints of the line segments. Include the first and last points
    # twice so we don't need any special syntax later to handle them.
    x = np.asarray(x)
    y = np.asarray(y)
    x_midpts = np.hstack((x[0], 0.5 * (x[1:] + x[:-1]), x[-1]))
    y_midpts = np.hstack((y[0], 0.5 * (y[1:] + y[:-1]), y[-1]))

    # Determine the start, middle, and end coordinate pair of each line segment.
    # Use the reshape to add an extra dimension so each pair of points is in its
    # own list. Then concatenate them to create:
    # [
    #   [(x1_start, y1_start), (x1_mid, y1_mid), (x1_end, y1_end)],
    #   [(x2_start, y2_start), (x2_mid, y2_mid), (x2_end, y2_end)],
    #   ...
    # ]
    coord_start = np.column_stack((x_midpts[:-1], y_midpts[:-1]))[:, np.newaxis, :]
    coord_mid = np.column_stack((x, y))[:, np.newaxis, :]
    coord_end = np.column_stack((x_midpts[1:], y_midpts[1:]))[:, np.newaxis, :]
    segments = np.concatenate((coord_start, coord_mid, coord_end), axis=1)

    lc = LineCollection(segments, **default_kwargs)
    lc.set_array(c)  # set the colors of each segment

    return ax.add_collection(lc)

def space_plot(x, y, xlabel, ylabel, color):
    '''
    Produce a 2D plot of a trajectory track with a line whose colour codes for time
    '''
    fig1, ax1 = plt.subplots()
    lines = colored_line(x, y, color, ax1, linewidth=2, cmap="plasma")
    fig1.colorbar(lines, label='time (ps)')  # add a color legend
    x_pad = (x.max() - x.min()) * 0.1
    y_pad = (y.max() - y.min()) * 0.1
    ax1.set_xlim(x.min()-x_pad, x.max()+x_pad)
    ax1.set_ylim(y.min()-y_pad, y.max()+y_pad)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

The `mdplus` Python package provides a PCA class designed specifically for use with trajectory data. It takes as input a trajectory in the form of an [n_frames, n_atoms, 3] coordinate array, and transforms it into an [n_frames, n_components] array, where n_components = min(n_frames, 3*n_atoms). Each frame of the trajectory is now a set of n_component *scores* - the coordinates of the snapshot in the PC space.

In [ ]:
from mdplus.pca import PCA
p = PCA() # Create an instance of the PCA transformer
scores = p.fit_transform(traj.xyz) # use it to transform the trajectory into the PC space

Now we can plot the trajectory in the 2D space defined by the first two PCs (first two eigenvectors):

In [ ]:
x = scores[:, 0] # The first coordinate of each snapshot in this space
y = scores[:, 1] # The second coordinate of each snapshot in this space
space_plot(x, y, 'PC0', 'PC1', traj.time)

The result is a visually intuitive representatioon of the trajectory as a "walk in shape space". It resolves a number of regions that are particularly densely sampled and could indicate low energy basins in shape space. 

But what are we missing by reducing the trajectory to motion in this 2D-plane? Projecting into the PC0/PC2 plane may give some insight into this:

In [ ]:
x = scores[:, 0] # The first coordinate of each snapshot in this space
y = scores[:, 2] # The third coordinate of each snapshot in this space
space_plot(x, y, 'PC0', 'PC2', traj.time)

This perspective on the distribution adds little to what was discernable in the PC0/PC1 plane. Roughly speaking, the trajectory involves the shape of the complex changing from one described by positive values of PC0 to ones described by negative values of PC0. But what sort of conformational change is this?

Each PC axis is a 3N-dimensional vector (eigenvector) that describes the collective motion of all atoms in the system. It can be useful to visualize these motions, which can be done by creating artificial trajectories for the system in which only motion along this PC is generated.

Let's do this for PC0:

In [ ]:
def make_animation_trajectory(traj, pca, ipc, smin, smax, n_frames):
    '''
    Produce an n_frame trajectory animating eigenvector ipc in pca from a value of smin to smax
    '''
    
    n_components = pca.n_components # The number of dimensions in the PC space
    # Create a short trajectory in PC space that samples the desired component over its full range:
    a = np.zeros((n_frames, n_components))
    a[:, ipc] = np.linspace(smin, smax, n_frames)
    # Perform the inverse transform to give the coordinates in Cartesian space
    xanim = pca.inverse_transform(a)
    # Turn this into an MTraj trajectory to view it:
    tanim = mdt.Trajectory(xanim, traj.topology)
    return tanim

smin = scores[0].min()
smax = scores[0].max()
tanim = make_animation_trajectory(traj, p, 0, smin, smax, 20)
view = nv.show_mdtraj(tanim)
view

The movie makes it clear that this principal component is dominated by motion of the last few amino acids at the C-terminal of the protein. Since there are well away from the ligand binding site, it's unlikeley they relate to anything that is functionally significant.

Let's remove these from consideration, and repeat the PCA process:

In [ ]:
sel = traj.topology.select('resid 0 to 260 or resname LIG') # everything except the last 7 C-terminal residues
clipped_traj = traj.atom_slice(sel) # Make a stripped-down version of the trajectory
clipped_p = PCA()
clipped_scores = clipped_p.fit_transform(clipped_traj.xyz)
x = clipped_scores[:, 0]
y = clipped_scores[:, 1]
space_plot(x, y, 'PC0', 'PC1', clipped_traj.time)

In [ ]:
# And now view the new PC0/PC2 plane:
x = clipped_scores[:, 0]
y = clipped_scores[:, 2]
space_plot(x, y, 'PC0', 'PC2', clipped_traj.time)

The depiction of shape-space is significantly different from before, but a "jumping amongst minima" form to the dynamics is still very evident. The view in the PC0/PC2 plane is now also helpful in further resolving conformational states.

Let's check what conformational variability the new PC0 now corresponds to:

In [ ]:
smin = clipped_scores[0].min()
smax = clipped_scores[0].max()
tanim = make_animation_trajectory(clipped_traj, clipped_p, 0, smin, smax, 20)
view = nv.show_mdtraj(tanim)
view

This mode of concerted motion particulaly features movement in regions either side of the ligand-binding cleft, so may well have functional significance. There seems no good reason to "edit" the structure of the system any further.

## 2. Clustering

Now that we have a method to visualize our trajectory as a walk in a low-dimensional version of a shape space that is (potentially) functionally relevant, we can move on to a more detailed exploration of methods to analyse the distribution of snapshots in this space, and particularly the way they cluster, as clusters may indicate the location of low-energy conformational states of the system.

There are many different clustering algorithms, they can all give potentially different results but none of them is automatically "best" for the analysis of MD data. Here we compare three common "agglomerative" methods: *single-linkage*, *complete-linkage*, and *ward's linkage*, with the alternative  *kmeans* method, all as implemented in the `scipy` Python package.

### 2.1 Comparison of agglomerative clustering methods

In [ ]:
from scipy.spatial.distance import pdist, cdist, squareform
from scipy.cluster.hierarchy import ward, fcluster, single, complete

All the agglomerative clustering methods take as input a distance matrix, which in this context is the matrix of rmsds between all snapshots in the trajectory. Because rmsd[i, j] == rmsd[j, i], and rmsd[i, i] = 0, we only need to calculate the upper triangle of the matrix, which saves memory:

In [ ]:
rmsdlist = []
for i in range(clipped_traj.n_frames - 1):
    rmsdlist.append(mdt.rmsd(clipped_traj[i+1:], clipped_traj[i]))
rmsds = np.concatenate(rmsdlist)

We will begin with single-linkage clustering. The process begins by performing the agglomeration, in which as a distance threshold is increased, individual points in the distribution gradually coalesce into a smaller and smaller number of larger and larger clusters, until all points are in a single cluster.

The optimal value of the distance threshold, which gives the "best" clustering, can in general only be identified by visual inspection and some trial-and-error. One method is the plot the number of clusters as a function of the distance threshold, and look for some sort of "natural break" in the distribution:

In [ ]:
cluster_method = single 
# Replace with:
# cluster_method = complete
# cluster method = ward
# to test other methods later
# Perform the clustering:
Z = cluster_method(rmsds)
# Graph the clustering performance:
plt.plot(Z[:, 2][::-1], range(999), 'o')
plt.ylabel('number of clusters')
plt.xlabel('distance threshold')

This is where it gets interactive. The aim is to identify a "natural break" in the agglomeration that identifies a "best" clustering of the data. In this case we see some evidence of this at a distance threshold of about 0.13. (When you look at alternative clustering methods you will likely see completely different values).

Plot the clustering this choice of cutoff generates:

In [ ]:
threshold = 0.132 # You may want to try changing this later
labels = fcluster(Z, threshold, criterion='distance')
# 'labels' is a vector of indices that assigns each snapshot to a cluster by id.
# By default the ids do not correspond to the order in the trajectory the clusters are found, but
# for convenience we transform them so they are:
label_order = []
for l in labels:
    if l not in label_order:
        label_order.append(l)
labels = np.array([label_order.index(l) for l in labels])

# Now group the snapshots in the PCA space according to the cluster they belong to:
clusters = [clipped_scores[labels == i] for i in set(labels)]

plt.figure(figsize=(10, 5))
plt.subplot(121)
for i, sc in enumerate(clusters):
    plt.plot(sc[:, 0], sc[:, 1], '.', label=f'cluster{i}')
plt.xlabel('PC0')
plt.ylabel('PC1')
plt.legend()
plt.subplot(122)
plt.plot(traj.time, labels, '*')
plt.xlabel('time (ps)')
plt.ylabel('cluster ID')

Single-linkage clustering suggests only three clusters and, at least by eye, they don't look particularly "natural".

Repeat the cells above after changing the clustering method to "complete" and then "ward".

Most likely you will conclude that "ward" gives the best-looking results - this is commonly the case for MD-type data, but not guaranteed!

### 2.2 Comparison with Kmeans clustering

Finally, let's look at *kmeans* clustering. Unlike the agglomerative methods, this approach requires the user to specify in advance the target number of clusters to generate. In additional contrast to the agglomerative methods, the immediate output is the set of cluster centroids, while the list of labels (the index of the cluster to which each snapshot belongs) must be calculated after.

In [ ]:
from scipy.cluster.vq import kmeans

n_clusters = 5 # A guess - other values should be explored as well
centroids, distortion = kmeans(scores, n_clusters)
plt.plot(scores[:, 0], scores[:, 1], '.', label='snapshots')
plt.plot(centroids[:, 0], centroids[:, 1], 'X', label='centroids')
plt.xlabel('PC0')
plt.ylabel('PC1')
plt.legend()

Now find the labels for each snapshot (the centroid the snapshot is closest to)

In [ ]:
dij = cdist(scores, centroids)
labels = dij.argmin(axis=1)
# Fix the label IDs so the first-encountered cluster is #1, the next #2, etc.:
label_order = []
for l in labels:
    if l not in label_order:
        label_order.append(l)
labels = np.array([label_order.index(l) for l in labels])

Now produce the usual figure:

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(121)
for i in range(5):
    mask = labels == i
    plt.plot(scores[mask, 0], scores[mask, 1], '.', label=f'cluster{i}')

plt.xlabel('PC0')
plt.ylabel('PC1')
plt.legend()
plt.subplot(122)
plt.plot(traj.time, labels, '*')
plt.xlabel('time (ps)')
plt.ylabel('cluster ID')

With a target of 5 clusters, the performance of ther kmeans method is not bad, but not as good as the Ward's linkage approach seems to be. Can you do better if you change the number of clusters?

# Summary

PCA is a very useful tool for producing representations of trajectories as a set of points in a low-dimensional shape space. Animations of the structure along an individual eigenvector can help determine if the structural deformation it captures is functionally relevant. Proceses such as relaxation and conformational sampling ("jumping among minima") can be visualized.

Once we have a trajectory represented as points in a shape space, we can use clustering methods to propose putative macrostates for the system, which may relate to underlying enertgy basins in the shape space. Representative conformations of each state can be generated, along with their populations. These metrics are useful tools for the evaluation of sampling and convergence.

Clustering is not an entirely objective process. The choice of clustering method, and parameters within it, can significantly affect the results obtained. Some level of user intervention and guidance is nearly always required.